# HTB Module 294 — Manipulating an ML Model

This offline study follows Section 3, **Manipulating the Model**. It reuses HTB's spam-classifier resources to compare two attack surfaces:

1. **Input manipulation:** change a message sent to an already trained classifier.
2. **Training-data poisoning:** change examples used to train a classifier, then see how its later predictions change.

The data and the example messages stay local. The link-looking text below is only a string passed to the classifier; this notebook never opens or requests it. This is an educational reproduction of HTB's toy dataset, not an assessment of a real mail service.

## How this notebook connects to earlier material

Module 292 introduced spam classification and Naive Bayes. This section reuses that model to show that a classifier's behavior depends on both its input and its training examples. We are not inventing a new classifier or moving on to a later attack category.

The course resource archive contains `main.py`, `train.csv`, `test.csv`, and `requirements.txt`. The original `main.py` runs training as soon as Python executes the file. To keep this notebook's steps explicit, we re-create its key functions in annotated cells and use the supplied CSVs. The code below retains the course's preprocessing, `CountVectorizer` settings, `MultinomialNB` classifier, and five-fold search over the same smoothing values.

## Probability notation used in this lesson

A simplified Naive Bayes score is:

\[
P(c \mid m) \propto P(c)\prod_{i=1}^{n}P(w_i \mid c)
\]

Read it aloud as: **“The probability of class c given message m is proportional to the probability of class c, times the product, from i equals one to n, of the probability of word i given class c.”**

- `c`, **“class c,”** is a candidate label: ham or spam.
- `m`, **“message m,”** is the message being classified.
- `P(c | m)`, **“the probability of class c given message m,”** is the model's score for a label after seeing the message.
- `P(c)`, **“the probability of class c,”** is its prior frequency in the training data.
- `w_i`, **“word sub i,”** is one token (or, in this model, sometimes a two-word phrase) in the message.
- `∏`, **“the product,”** means multiply the listed values.
- `∝`, **“is proportional to,”** means these are scores before normalization into probabilities.

The “naive” assumption is that words contribute independently once we condition on the class. That is a simplifying model assumption, not a claim that real language has no word relationships. The predicted label is `arg max_c P(c | m)`, read **“arg max over c of the probability of c given m”**: choose the class with the largest score.

Because multiplying many small probabilities is inconvenient numerically, the equivalent log score is `log P(c) + sum_i log P(w_i | c)`. Read `sum_i` as **“sum over i.”** Taking logs turns multiplication into addition; each word adds evidence for or against a class. Enough ham-associated tokens can therefore outweigh spam-associated tokens in this particular model.

## 1. Load the HTB-provided files

The path lookup supports the two common ways to open this notebook: with the working directory set to this folder, or with the repository root as the working directory. It does not download data or contact a service.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

# Find the resource folder whether Jupyter starts here or at the repository root.
resource_candidates = [
    Path.cwd() / "resources",
    Path.cwd() / "AI_Red_Teaming" / "ML_Model_Manipulation" / "resources",
]
RESOURCE_DIR = next((path for path in resource_candidates if path.is_dir()), None)
if RESOURCE_DIR is None:
    raise FileNotFoundError(
        "Could not find resources/train.csv and resources/test.csv. "
        "Open this notebook from the repository or its own folder."
    )

TRAIN_PATH = RESOURCE_DIR / "train.csv"
TEST_PATH = RESOURCE_DIR / "test.csv"

# Keep the supplied source nearby for comparison, but do not import it: its
# top-level main block starts training immediately when the file is imported.
HTB_SOURCE_PATH = RESOURCE_DIR / "main.py"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

# A small inventory tells us the shape and label balance without dumping SMS text.
resource_summary = pd.DataFrame(
    {
        "dataset": ["train", "test"],
        "rows": [len(train_raw), len(test_raw)],
        "columns": [", ".join(train_raw.columns), ", ".join(test_raw.columns)],
        "labels": [
            train_raw["label"].value_counts().to_dict(),
            test_raw["label"].value_counts().to_dict(),
        ],
    }
)
display(resource_summary)

# Defensive checks make schema mismatches obvious before training begins.
required_columns = {"label", "message"}
assert required_columns.issubset(train_raw.columns)
assert required_columns.issubset(test_raw.columns)
assert set(train_raw["label"].dropna().unique()).issubset({"ham", "spam"})
assert set(test_raw["label"].dropna().unique()).issubset({"ham", "spam"})

## 2. Match HTB's text preprocessing

Before counting words, HTB's code lowercases text, removes most punctuation and digits, tokenizes it, removes common stop words, and stems words. **Stemming** reduces related word forms to a shared rough root—for example, “claiming” may be reduced toward “claim.” The list subtracts four words (`free`, `win`, `cash`, `urgent`) from the stop-word list so they remain available as potentially useful spam clues.

The tokenizer and stop-word lists are NLTK data resources. This cell checks for them but does not automatically download anything. If they are missing, use the environment already prepared for this HTB resource or arrange the corpus files separately.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Fail early with a clear message rather than failing halfway through training.
try:
    STOP_WORDS = set(stopwords.words("english")) - {"free", "win", "cash", "urgent"}
    _ = word_tokenize("checking the local tokenizer resources")
except LookupError as exc:
    raise RuntimeError(
        "NLTK is installed, but its English stopwords or tokenizer data are missing. "
        "This notebook intentionally does not download corpora automatically. "
        "Use the configured AI/HTB environment with the required NLTK data installed."
    ) from exc

STEMMER = PorterStemmer()

def preprocess_message(message: str) -> str:
    """Mirror the HTB resource's lowercasing, filtering, tokenizing, and stemming."""
    message = str(message).lower()

    # Keep lowercase letters, whitespace, dollar signs, and exclamation marks,
    # matching the expression in HTB's main.py. Other characters are removed.
    message = re.sub(r"[^a-z\s$!]", "", message)

    # Stop-word removal reduces common words; stemming groups related forms.
    tokens = word_tokenize(message)
    tokens = [STEMMER.stem(word) for word in tokens if word not in STOP_WORDS]
    return " ".join(tokens)

def prepare_training_frame(raw_frame: pd.DataFrame) -> pd.DataFrame:
    """Return a cleaned copy; never mutate the original resource DataFrame."""
    prepared = raw_frame[["label", "message"]].copy()
    prepared["label"] = prepared["label"].str.lower().str.strip()
    prepared["message"] = prepared["message"].map(preprocess_message)

    # HTB removes duplicate rows after preprocessing, so identical cleaned
    # examples do not gain extra influence merely by being repeated.
    prepared = prepared.drop_duplicates().reset_index(drop=True)
    return prepared

# A tiny example demonstrates the transformation without showing a whole dataset.
preprocess_message("Congratulations! You won a prize.")

## 3. Train and inspect the baseline classifier

`CountVectorizer` converts text into a table of token counts. With `ngram_range=(1, 2)`, it counts both single words (unigrams) and adjacent two-word phrases (bigrams). `MultinomialNB` is the Naive Bayes model commonly used with word counts.

`GridSearchCV` tries three smoothing values for `alpha` and uses five folds. **Smoothing** avoids assigning a zero probability to a word that did not appear with a class in one training fold. The search chooses the value with the best average F1 score. F1 balances precision and recall; we will still separately measure held-out accuracy because HTB reports it.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline


def train_classifier(raw_frame: pd.DataFrame):
    """Train the HTB-style model from a raw label/message DataFrame."""
    prepared = prepare_training_frame(raw_frame)
    messages = prepared["message"]

    # Convert labels to the numeric class IDs used in the HTB example:
    # ham = class 0; spam = class 1.
    labels = prepared["label"].map({"ham": 0, "spam": 1}).to_numpy()

    # Formula connection: the vectorizer turns each message into counts of w_i.
    # Bigrams let a phrase contribute evidence alongside its individual words.
    pipeline = Pipeline(
        steps=[
            (
                "vectorizer",
                CountVectorizer(min_df=1, max_df=0.9, ngram_range=(1, 2)),
            ),
            ("classifier", MultinomialNB()),
        ]
    )

    # Each candidate alpha is evaluated by five-fold cross-validation using F1.
    # n_jobs=1 keeps this small teaching example on one CPU worker.
    search = GridSearchCV(
        estimator=pipeline,
        param_grid={"classifier__alpha": [0.1, 0.5, 1.0]},
        cv=5,
        scoring="f1",
        n_jobs=1,
    )
    search.fit(messages, labels)
    return search.best_estimator_, prepared, search


def message_probabilities(model, messages):
    """Return predicted labels and class probabilities for one or more messages."""
    if isinstance(messages, str):
        messages = [messages]

    # Apply the same preprocessing at inference that we used before training.
    cleaned_messages = [preprocess_message(message) for message in messages]

    # Pipeline applies CountVectorizer, then the trained Naive Bayes classifier.
    probabilities = model.predict_proba(cleaned_messages)
    predicted_ids = model.classes_[np.argmax(probabilities, axis=1)]
    predicted_labels = np.where(predicted_ids == 0, "ham", "spam")
    return predicted_labels, probabilities


def accuracy_on_test(model, raw_test_frame: pd.DataFrame) -> float:
    """Compute correct predictions divided by all held-out test examples."""
    true_ids = raw_test_frame["label"].map({"ham": 0, "spam": 1}).to_numpy()
    predicted_labels, _ = message_probabilities(model, raw_test_frame["message"].tolist())
    predicted_ids = np.where(predicted_labels == "ham", 0, 1)

    # Accuracy = (number correctly classified) / (number evaluated).
    return float(np.mean(predicted_ids == true_ids))

baseline_model, baseline_train_prepared, baseline_search = train_classifier(train_raw)
baseline_accuracy = accuracy_on_test(baseline_model, test_raw)

print(f"Baseline held-out accuracy: {baseline_accuracy * 100:.2f}%")
print(f"Chosen smoothing alpha: {baseline_model.named_steps['classifier'].alpha}")
print(f"Rows after preprocessing and duplicate removal: {len(baseline_train_prepared)}")

## 4. Input manipulation: change the message, keep the model fixed

Here the trained `baseline_model` does not change. We vary only the text sent to it. The probability table makes the model's reaction visible. In HTB's run, the original prize message is classified as spam with 100% reported probability. The exact results depend on the supplied data, preprocessing resources, and library versions.

In [ ]:
# This is inert text for the local classifier. No URL is opened or requested.
original_spam_message = (
    "Congratulations! You won a prize. Click here to claim: "
    "https://bit.ly/3YCN7PF"
)

rephrasing_examples = [
    "Congratulations!",
    "Congratulations! You won a prize.",
    "Click here to claim: https://bit.ly/3YCN7PF",
    "https://bit.ly/3YCN7PF",
    "Your account has been blocked. You can unlock your account in the next 24h: "
    "https://bit.ly/3YCN7PF",
    original_spam_message,
]

predicted_labels, probability_matrix = message_probabilities(
    baseline_model, rephrasing_examples
)

# model.classes_ is [0, 1], which we defined as [ham, spam].
rephrasing_results = pd.DataFrame(
    {
        "message": rephrasing_examples,
        "predicted label": predicted_labels,
        "ham probability (%)": np.round(probability_matrix[:, 0] * 100, 2),
        "spam probability (%)": np.round(probability_matrix[:, 1] * 100, 2),
    }
)
display(rephrasing_results)

### Read the probability table carefully

This experiment is an example of **input manipulation (ML01)**: the classifier is fixed, but the input wording changes. “Rephrasing” is a way to probe which patterns matter to this model. The probabilities describe this particular model's scores; they do not tell us whether a message is actually safe, truthful, or legitimate.

The HTB page's example account-warning wording barely flips the decision to ham (57.39% ham versus 42.61% spam). This is a narrow toy-model result, not a reliable recipe against other filters. Modern systems may use different training data, text normalization, additional signals, and layered detection.

## 5. Input manipulation by overpowering

The next experiment appends ordinary-looking text to the original spam. Under Naive Bayes, each token contributes evidence. In log space, the model adds that evidence, so many ham-associated tokens can overwhelm a smaller number of spam-associated tokens.

HTB uses the start of a Lorem Ipsum passage. This notebook includes the passage as plain text only. The model does not visit a web page or perform any action with it.

In [ ]:
benign_overpowering_text = (
    "But I must explain to you how all this mistaken idea of denouncing pleasure "
    "and praising pain was born and I will give you a complete account of the system, "
    "and expound the actual teachings of the great explorer of the truth, "
    "the master-builder of human happiness."
)

overpowered_message = original_spam_message + " " + benign_overpowering_text

overpower_labels, overpower_probs = message_probabilities(
    baseline_model,
    [original_spam_message, overpowered_message],
)

overpower_results = pd.DataFrame(
    {
        "case": ["original message", "same message plus benign text"],
        "predicted label": overpower_labels,
        "ham probability (%)": np.round(overpower_probs[:, 0] * 100, 2),
        "spam probability (%)": np.round(overpower_probs[:, 1] * 100, 2),
    }
)
display(overpower_results)

The security lesson is not “longer text is always safer.” It is that a simple bag-of-words model can be influenced by token evidence without understanding which parts of a message are visible, relevant, or trustworthy. HTB notes that hidden HTML comments could contain words the victim does not see while a naive classifier still processes them. A defender should parse and normalize content consistently, remove hidden or irrelevant markup, and treat a classifier result as one signal rather than the whole decision.

## 6. Data poisoning: first create the deliberately small training set

HTB's shell command `head -n 101 train.csv > poison.csv` copies the header plus the first 100 examples. In pandas, `train_raw.head(100)` means the first 100 data rows; the header is already represented by the DataFrame's column names.

The test set stays untouched. This matters because we want to ask two separate questions: did the model change its target-message prediction, and how did it perform on examples held out from training?

In [ ]:
small_train_raw = train_raw.head(100).copy()
small_model, small_train_prepared, small_search = train_classifier(small_train_raw)
small_accuracy = accuracy_on_test(small_model, test_raw)

# This is the HTB target message used in the poisoning experiment.
target_message = "Hello World! How are you doing?"
small_target_labels, small_target_probs = message_probabilities(
    small_model, target_message
)

print(f"Small-set held-out accuracy: {small_accuracy * 100:.2f}%")
print(f"Target prediction before poisoning: {small_target_labels[0]}")
print(
    "Target probabilities before poisoning: "
    f"ham={small_target_probs[0, 0] * 100:.2f}%, "
    f"spam={small_target_probs[0, 1] * 100:.2f}%"
)

The HTB output reports 94.4% accuracy for this smaller set and says the target message is ham with 98.7% probability. The smaller sample is less representative of the full training distribution, so a small number of added rows can have more influence. We should not confuse this intentionally sensitive setup with the likely behavior of a model trained on a much larger, carefully curated dataset.

## 7. Add targeted poisoned examples

Now we add two rows to the **training copy** and label them spam. This is a small, targeted poisoning demonstration: it tries to change the model's answer for one chosen phrase. The held-out test file is still unchanged.

In [ ]:
first_poison_rows = pd.DataFrame(
    {
        "label": ["spam", "spam"],
        "message": ["Hello World", "How are you doing?"],
    }
)

poisoned_train_raw = pd.concat(
    [small_train_raw, first_poison_rows],
    ignore_index=True,
)
poisoned_model, poisoned_train_prepared, poisoned_search = train_classifier(
    poisoned_train_raw
)
poisoned_accuracy = accuracy_on_test(poisoned_model, test_raw)
poisoned_labels, poisoned_probs = message_probabilities(
    poisoned_model, target_message
)

print(f"Rows before poisoning (raw): {len(small_train_raw)}")
print(f"Rows after adding the two examples (raw): {len(poisoned_train_raw)}")
print(f"Rows after preprocessing and duplicate removal: {len(poisoned_train_prepared)}")
print(f"Held-out accuracy after poisoning: {poisoned_accuracy * 100:.2f}%")
print(f"Target prediction after poisoning: {poisoned_labels[0]}")
print(
    "Target probabilities after poisoning: "
    f"ham={poisoned_probs[0, 0] * 100:.2f}%, "
    f"spam={poisoned_probs[0, 1] * 100:.2f}%"
)

HTB reports that the target flips from ham to spam, reaching 79.66% spam probability after the first two examples. The key point is the targeted effect: the model learned that pieces of this specific benign message are associated with the spam label. A high overall score on unrelated examples would not prove that this targeted behavior is harmless.

## 8. Add overlapping phrases and compare the trade-off

HTB adds two overlapping phrases as additional spam examples. They are not byte-for-byte duplicates of one another, so the preprocessing and duplicate-removal step keeps them. The following cell retrains on that new training copy and compares the target probabilities and held-out accuracy.

In [ ]:
additional_poison_rows = pd.DataFrame(
    {
        "label": ["spam", "spam"],
        "message": [
            "Hello World! How are you",
            "World! How are you doing?",
        ],
    }
)

more_poisoned_train_raw = pd.concat(
    [poisoned_train_raw, additional_poison_rows],
    ignore_index=True,
)
more_poisoned_model, more_poisoned_train_prepared, more_poisoned_search = train_classifier(
    more_poisoned_train_raw
)
more_poisoned_accuracy = accuracy_on_test(more_poisoned_model, test_raw)
more_poisoned_labels, more_poisoned_probs = message_probabilities(
    more_poisoned_model, target_message
)

# The source removes exact duplicate rows, so adding the same row again would
# not increase its count after preparation.
exact_duplicate_example = pd.concat(
    [more_poisoned_train_raw, more_poisoned_train_raw.tail(1)],
    ignore_index=True,
)
prepared_with_duplicate = prepare_training_frame(exact_duplicate_example)
prepared_without_duplicate = prepare_training_frame(more_poisoned_train_raw)

comparison = pd.DataFrame(
    {
        "training scenario": ["first 100 rows", "plus two targeted rows", "plus two overlapping rows"],
        "prepared rows": [
            len(small_train_prepared),
            len(poisoned_train_prepared),
            len(more_poisoned_train_prepared),
        ],
        "held-out accuracy (%)": [
            round(small_accuracy * 100, 2),
            round(poisoned_accuracy * 100, 2),
            round(more_poisoned_accuracy * 100, 2),
        ],
        "target prediction": [
            small_target_labels[0],
            poisoned_labels[0],
            more_poisoned_labels[0],
        ],
        "target spam probability (%)": [
            round(small_target_probs[0, 1] * 100, 2),
            round(poisoned_probs[0, 1] * 100, 2),
            round(more_poisoned_probs[0, 1] * 100, 2),
        ],
    }
)
display(comparison)

print(
    "Prepared-row count with an extra exact duplicate: "
    f"{len(prepared_with_duplicate)}"
)
print(f"Prepared-row count without that duplicate: {len(prepared_without_duplicate)}")

HTB reports 99.6% spam probability for the target after the two overlapping phrases, with 94.0% held-out accuracy. Compared with 94.4% before the targeted rows, the accuracy change is 0.4 **percentage points**: `94.4 - 94.0 = 0.4`. Percentage points are the direct subtraction of two percentages. The attack changes a selected prediction much more than it changes this aggregate metric.

Your exact values may differ slightly with package versions or the installed NLTK data. Read this as an experiment in cause and effect, not as a universal score. In production, training data is generally much larger, and a fixed number of inserted rows usually has less influence. The attacker's access to the data collection or training pipeline is also a required assumption.

## 9. What this implementation teaches—and what defenders should check

**HTB lesson:** a model can react strongly to selected input words, and a deliberately small training sample can be shifted by a few carefully chosen labeled examples. The experiment uses a classic bag-of-words model whose assumptions make these effects easy to observe.

**Connection to prior material:** the Naive Bayes posterior combines a class prior with token likelihoods. Input manipulation changes the tokens being scored. Data poisoning changes the token/class evidence learned from training rows.

**Defensive interpretation:** verify where training data came from, keep provenance and immutable audit records, validate label quality, deduplicate and inspect suspicious examples, and evaluate targeted cases as well as aggregate metrics. Monitor per-class precision/recall and slices of behavior so a localized failure is not hidden by a high overall accuracy. At inference, normalize and sanitize text consistently and treat the model score as evidence rather than authorization.

This notebook stays within this section: it does not add a different attack algorithm, does not send inputs to a remote model, and does not use or target a live service.